# EDA: State Transition Log

## Setup
Reset database and seed base data.

In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 5d58187c-1983-406b-9a35-f1256cfd14de
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: 6bbd93a7-40e4-4922-9756-de8432686b27
Seeded SystemPrompt 'format' with ID: 1 and GUID: 687ac1e6-a57b-44b2-bad9-70e56d657a80
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: fcbd7222-6144-4fd1-85fb-0b9fa9af6a0c
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [3]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.enums.logging_enums import PROVIDER_TYPE
from app.enums.system_enums import SYSTEM_TYPE
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/session_linked_execution.py")
file.write_text("def ping( user ):\n return f\"pong {user}\"")

# Construct input using session metadata
input_data = {
    "file_path":  str(file),
    "session_id": session_row.id,
    "system":     SYSTEM_TYPE.LINTING.value
}

# Run associated program
program = ProgramProviderFactory.create(
    id=session_row.program_provider_id,
    called_by_type=PROVIDER_TYPE.SESSION,
    called_by_id=session_row.id
)
result = program.run(input_data, session_id=session_row.id)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))


✅ Final Program Result (via SessionConfig):
{
  "state": "end",
  "previous_state": "preprocessing",
  "state_type": "end",
  "decision": "accept",
  "steps": 2,
  "max_steps": 20,
  "summary": "preprocessing complete",
  "output": {
    "state": "end",
    "file_path": "working_files\\final_prog_1753403055.py",
    "session_id": 1,
    "system": "linting",
    "reason": "preprocessing complete",
    "steps": 2,
    "retry_count": 0,
    "_last_state": "preprocessing",
    "decision": "accept",
    "score": null,
    "state_output": {
      "state": "end",
      "previous_state": "preprocess",
      "state_type": "end",
      "decision": "accept",
      "steps": 2,
      "max_steps": 10,
      "summary": "Completed",
      "provider_name": "preprocessing_controller"
    }
  },
  "provider_name": "codecritic_program"
}


### Load raw log entries

In [6]:
import sqlite3
import pandas as pd
import json
from app.db.connection import DB_PATH

# Connect and load transition logs
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM state_transition_log WHERE session_id='1'", conn)
conn.close()

# Sort and format
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(by='timestamp').reset_index(drop=True)

# Display trace
for i, row in df.iterrows():
    print(f"\n🔹 Transition {i + 1} — ID {row['id']}")
    print(f"🕒 Timestamp: {row['timestamp']}")
    print(f"📘 From: {row['from_state']}  ➡️  To: {row['to_state']}")
    print(f"🧠 Reason: {row.get('reason') or '—'}")
    print(f"🧾 Decision: {row.get('decision') or '—'}")
    print(f"👤 Triggered by: {row.get('triggered_by') or '—'}")
    print(f"📍 Step #: {row.get('step') if row.get('step') is not None else '—'}")
    print(f"🏷️ Entity: {row['entity_type']} (ID {row['entity_id']})")

    if row.get("input_snapshot_id") or row.get("output_snapshot_id"):
        print(f"📥 Input Snapshot: {row.get('input_snapshot_id') or '—'}")
        print(f"📤 Output Snapshot: {row.get('output_snapshot_id') or '—'}")

    try:
        metadata = json.loads(row["transition_metadata"]) if row["transition_metadata"] else {}
        pretty_meta = json.dumps(metadata, indent=2)
        print("🧳 Metadata:")
        print(pretty_meta if pretty_meta else "  —")
    except Exception:
        print("🧳 Metadata (raw):")
        print(row["transition_metadata"] or "  —")

    print("—" * 80)



🔹 Transition 1 — ID 1
🕒 Timestamp: 2025-06-07 22:53:24.539500+00:00
📘 From: start  ➡️  To: preprocessing
🧠 Reason: start of program
🧾 Decision: STATE_DECISION_TYPE.REJECT
👤 Triggered by: codecritic_program
📍 Step #: 1
🏷️ Entity: PROVIDER_TYPE.PROGRAM (ID 1)
🧳 Metadata:
{}
————————————————————————————————————————————————————————————————————————————————

🔹 Transition 2 — ID 2
🕒 Timestamp: 2025-06-07 22:53:24.550541+00:00
📘 From: start  ➡️  To: preprocess
🧠 Reason: kickoff
🧾 Decision: STATE_DECISION_TYPE.REJECT
👤 Triggered by: preprocessing_controller
📍 Step #: 2
🏷️ Entity: PROVIDER_TYPE.CONTROLLER (ID 1)
🧳 Metadata:
{}
————————————————————————————————————————————————————————————————————————————————

🔹 Transition 3 — ID 3
🕒 Timestamp: 2025-06-07 22:53:24.555133+00:00
📘 From: start  ➡️  To: code_stability
🧠 Reason: initial stability check
🧾 Decision: STATE_DECISION_TYPE.REJECT
👤 Triggered by: linting_system_provider
📍 Step #: 3
🏷️ Entity: PROVIDER_TYPE.SYSTEM (ID 1)
🧳 Metadata:
{}
———————

### Parse enum fields

In [ ]:
from app.enums import logging_enums, fsm_enums, agent_enums
print('Columns:', df.columns.tolist())

### Validate field values

In [ ]:
print(df.isnull().sum())

### Basic counts

In [ ]:
print(df.shape)
print(df['timestamp'].min(), df['timestamp'].max())